# Bridge route preference

Causal MSE (× 1e3), mean ± SE over 10 data seeds (3 repeats averaged within each seed). Both bridges are correctly specified; each scenario stresses one route. Oracle rows use the closed-form bridge in place of the learned one.

**Protocol.** No cross-fitting: 25% of the sample is held out as the estimation fold and both bridges are trained on the other 75%. The two settings differ in how the training rows are split between the two stages — *overlapping*: stage 1 = first 75% of them, stage 2 = last 75% (so the stages share half); *independent*: disjoint halves. Cross-fitting is deliberately excluded here, since its own variance reduction would confound the split comparison.

In [1]:
import glob
import numpy as np
import pandas as pd

ORDER = ["OutcomeNet", "TreatmentNet", "DRPCLNET", "OracleOM", "OracleTM"]
NAMES = {"OutcomeNet": "OutcomeNet", "TreatmentNet": "TreatmentNet", "DRPCLNET": "DR-PCL-Net",
         "OracleOM": "Oracle outcome bridge", "OracleTM": "Oracle treatment bridge"}
SEEDS, REPEATS = range(10), 3

def mse_table(folder, level, descending=False):
    files = glob.glob("../Results/BridgeRoutePreference/%s/*.csv" % folder)
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d = d[d.data_seed.isin(SEEDS) & d.Algorithm.isin(ORDER)]
    d[level] = d[level].round(2)
    d = d.drop_duplicates(["Algorithm", level, "data_seed", "train_seed"], keep="last")
    runs = d[d.Algorithm == "OutcomeNet"].groupby(level).size()
    assert (runs == REPEATS * len(SEEDS)).all(), "incomplete grid: %s" % runs.to_dict()
    g = d.groupby(["Algorithm", level, "data_seed"]).Causal_MSE.mean().groupby(["Algorithm", level])
    m, se = g.mean() * 1e3, g.std() / np.sqrt(g.size()) * 1e3          # SE over data seeds
    t = (m.map("{:.2f}".format) + " ± " + se.map("{:.2f}".format)).unstack(level).reindex(ORDER)
    t = t[sorted(t.columns, reverse=descending)]
    t.index = [NAMES[a] for a in t.index]
    return t

def show(prefix, level, descending=False, suffix=""):
    for n in (2000, 4000):
        for split in ("overlap", "independent"):
            print("n = %d, %s split" % (n, "overlapping" if split == "overlap" else "independent"))
            print(mse_table("%s_n%d_%s%s" % (prefix, n, split, suffix), level, descending).to_string()); print()

## Scenario B: treatment-route stress

Treatment assignment depends increasingly on a latent component (`kappa_S`); the outcome-bridge problem is held fixed.

In [2]:
show("scenarioB", "kappa_s")

n = 2000, overlapping split


kappa_s                          0.0          0.2           0.4           0.6
OutcomeNet               5.70 ± 0.79  6.24 ± 0.94   8.51 ± 1.32  16.25 ± 2.30
TreatmentNet             6.50 ± 0.79  7.22 ± 0.95  12.79 ± 1.60  36.80 ± 4.42
DR-PCL-Net               3.53 ± 0.33  4.65 ± 0.54   7.29 ± 0.85  15.65 ± 1.85
Oracle outcome bridge    2.98 ± 0.48  2.98 ± 0.48   2.98 ± 0.48   2.98 ± 0.48
Oracle treatment bridge  4.32 ± 0.31  5.52 ± 0.61   8.55 ± 1.02  25.17 ± 2.66

n = 2000, independent split


kappa_s                          0.0           0.2           0.4           0.6
OutcomeNet               9.19 ± 1.12  10.14 ± 1.61  15.70 ± 2.61  34.44 ± 4.14
TreatmentNet             6.75 ± 1.14   8.23 ± 1.45  14.14 ± 2.02  39.99 ± 5.72
DR-PCL-Net               3.86 ± 0.43   5.09 ± 0.66   8.52 ± 1.16  19.14 ± 2.47
Oracle outcome bridge    2.98 ± 0.48   2.98 ± 0.48   2.98 ± 0.48   2.98 ± 0.48
Oracle treatment bridge  4.32 ± 0.31   5.52 ± 0.61   8.55 ± 1.02  25.17 ± 2.66

n = 4000, overlapping split


kappa_s                          0.0          0.2          0.4           0.6
OutcomeNet               3.80 ± 0.52  4.03 ± 0.61  4.88 ± 0.70   9.69 ± 1.08
TreatmentNet             4.78 ± 0.41  5.00 ± 0.85  7.97 ± 1.23  27.39 ± 3.31
DR-PCL-Net               2.40 ± 0.32  2.71 ± 0.31  4.17 ± 0.55   8.91 ± 1.27
Oracle outcome bridge    2.34 ± 0.35  2.34 ± 0.35  2.34 ± 0.35   2.34 ± 0.35
Oracle treatment bridge  2.55 ± 0.27  2.85 ± 0.43  4.37 ± 0.56  11.32 ± 1.17

n = 4000, independent split


kappa_s                          0.0          0.2          0.4           0.6
OutcomeNet               5.77 ± 0.83  7.22 ± 1.17  9.26 ± 1.82  21.69 ± 3.83
TreatmentNet             5.30 ± 0.56  5.57 ± 0.79  8.97 ± 1.20  28.40 ± 3.43
DR-PCL-Net               2.56 ± 0.32  2.94 ± 0.31  4.52 ± 0.51  12.23 ± 2.12
Oracle outcome bridge    2.34 ± 0.35  2.34 ± 0.35  2.34 ± 0.35   2.34 ± 0.35
Oracle treatment bridge  2.55 ± 0.27  2.85 ± 0.43  4.37 ± 0.56  11.32 ± 1.17



## Scenario O: outcome-route stress

The outcome-proxy reliability `lambda_WS` decreases (stress increases left to right); the treatment-bridge problem is held fixed by pinning `lambda_ZS * lambda_WS`, so the exact treatment bridge — which uses only the confounder proxy — is invariant along the sweep and its oracle row is constant.

The latent is drawn BEFORE its proxies (the textbook order), so a fixed seed holds the latents and the outcome fixed and only the S-proxies move.

In [3]:
show("scenarioO_compensated", "lambda_ws", descending=True, suffix="_forward")

n = 2000, overlapping split


lambda_ws                        0.9          0.7          0.6          0.5           0.4
OutcomeNet               3.25 ± 0.55  3.69 ± 0.62  5.34 ± 1.11  9.13 ± 1.07  15.49 ± 2.22
TreatmentNet             3.45 ± 0.40  3.66 ± 0.40  4.21 ± 0.51  4.69 ± 0.58   5.57 ± 0.68
DR-PCL-Net               2.56 ± 0.26  4.00 ± 0.68  4.78 ± 0.58  6.46 ± 0.42  10.97 ± 1.50
Oracle outcome bridge    1.56 ± 0.26  1.95 ± 0.38  3.10 ± 0.83  4.83 ± 0.97   7.49 ± 1.54
Oracle treatment bridge  2.92 ± 0.34  2.92 ± 0.34  2.92 ± 0.34  2.92 ± 0.34   2.92 ± 0.34

n = 2000, independent split


lambda_ws                        0.9          0.7           0.6           0.5           0.4
OutcomeNet               7.77 ± 1.88  8.14 ± 1.98  10.14 ± 1.70  15.08 ± 1.75  20.56 ± 1.85
TreatmentNet             3.89 ± 0.59  4.28 ± 0.53   4.88 ± 0.64   5.36 ± 0.70   7.27 ± 0.78
DR-PCL-Net               2.63 ± 0.33  4.38 ± 0.96   5.27 ± 0.69   7.23 ± 0.69  11.10 ± 1.76
Oracle outcome bridge    1.56 ± 0.26  1.95 ± 0.38   3.10 ± 0.83   4.83 ± 0.97   7.49 ± 1.54
Oracle treatment bridge  2.92 ± 0.34  2.92 ± 0.34   2.92 ± 0.34   2.92 ± 0.34   2.92 ± 0.34

n = 4000, overlapping split


lambda_ws                        0.9          0.7          0.6          0.5          0.4
OutcomeNet               1.26 ± 0.07  1.98 ± 0.16  2.60 ± 0.25  4.76 ± 0.79  7.60 ± 1.08
TreatmentNet             1.59 ± 0.16  1.59 ± 0.19  1.94 ± 0.22  2.40 ± 0.25  2.94 ± 0.36
DR-PCL-Net               0.76 ± 0.06  1.49 ± 0.13  2.41 ± 0.20  3.20 ± 0.29  5.06 ± 0.42
Oracle outcome bridge    0.68 ± 0.12  0.90 ± 0.12  1.24 ± 0.16  2.30 ± 0.32  3.48 ± 0.64
Oracle treatment bridge  1.13 ± 0.10  1.13 ± 0.10  1.13 ± 0.10  1.13 ± 0.10  1.13 ± 0.10

n = 4000, independent split


lambda_ws                        0.9          0.7          0.6          0.5          0.4
OutcomeNet               2.33 ± 0.29  3.35 ± 0.35  4.23 ± 0.77  5.93 ± 1.14  9.57 ± 1.72
TreatmentNet             1.74 ± 0.18  1.80 ± 0.20  2.18 ± 0.20  2.69 ± 0.24  3.36 ± 0.45
DR-PCL-Net               0.82 ± 0.07  1.57 ± 0.17  2.53 ± 0.21  3.43 ± 0.27  5.32 ± 0.43
Oracle outcome bridge    0.68 ± 0.12  0.90 ± 0.12  1.24 ± 0.16  2.30 ± 0.32  3.48 ± 0.64
Oracle treatment bridge  1.13 ± 0.10  1.13 ± 0.10  1.13 ± 0.10  1.13 ± 0.10  1.13 ± 0.10

